# 03 · Task 1 baseline — TF-IDF + Logistic Regression
**Owner:** C (Task-1 lead) · **Phase 1** (runs in seconds on CPU)

Protocol: fit on `train`, choose C on `val`, evaluate ONCE on `test` (Bitext) and on the real-customer
external test (BANKING77, mapped intents). The saved model is also the Agentic RAG's intent tool.

In [1]:
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # Colab: clone the team repo once per session, install requirements, load API keys from Secrets
    REPO_URL = "https://github.com/sh1or1tama1/banking-support-agent.git"
    if not os.path.exists("/content/banking-support-agent"):
        subprocess.run(["git", "clone", REPO_URL, "/content/banking-support-agent"], check=True)
    os.chdir("/content/banking-support-agent")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import userdata
    for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
sys.path.insert(0, os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))

import joblib, pandas as pd
from IPython.display import display
from src import config as C
from src.utils import set_seed
from src.data.load import load_banking77
from src.data.eval_sets import banking77_mapped
from src.classify import tfidf_lr, metrics as M
set_seed()

df = pd.read_parquet(C.SPLITS_FILE)
train, val, test = (df[df["split"] == s] for s in ("train", "val", "test"))
b77_ext = banking77_mapped(load_banking77())                # high+medium confidence mappings
b77_strict = banking77_mapped(load_banking77(), strict=True)  # high confidence only
print(len(train), len(val), len(test), "| external:", len(b77_ext), "(strict", len(b77_strict), ")")

17882 3832 3831 | external: 520 (strict 240 )


In [2]:
model, grid = tfidf_lr.tune_and_fit(train, val)
display(grid)
joblib.dump(model, C.OUTPUTS / "tfidf_lr.joblib")

,C,val_accuracy,val_macro_f1
0,0.1,0.997651,0.997674
1,1.0,0.999478,0.999482
2,10.0,0.999478,0.999484
3,100.0,0.999739,0.999741


['/Users/fangziming/Desktop/Uchicago课程/Multimodal/banking-support-agent/outputs/tfidf_lr.joblib']

In [3]:
results = []
test = test.copy()
test["pred"], ms = tfidf_lr.predict_timed(model, test["instruction"].tolist())
results.append(M.summary(test["intent"], test["pred"], "tfidf_lr", "bitext_test", ms))
for name, ext in (("banking77_mapped", b77_ext), ("banking77_mapped_strict", b77_strict)):
    p, ms = tfidf_lr.predict_timed(model, ext["text"].tolist())
    results.append(M.summary(ext["gold_intent"], p, "tfidf_lr", name, ms))
res = pd.DataFrame(results); display(res)
res.to_csv(C.TABLES / "task1_tfidf_lr_results.csv", index=False)

,model,eval_set,n,accuracy,macro_f1,ms_per_query
0,tfidf_lr,bitext_test,3831,0.9992,0.9991,0.038
1,tfidf_lr,banking77_mapped,520,0.4096,0.5158,0.050
2,tfidf_lr,banking77_mapped_strict,240,0.5542,0.6302,0.048


In [4]:
display(M.per_class(test["intent"], test["pred"]).head(8))
display(M.accuracy_by_flag(test))
display(M.top_confusions(test["intent"], test["pred"]))
M.plot_confusion(test["intent"], test["pred"], "task1_tfidf_confusion_bitext", "TF-IDF + LR · Bitext test")

,precision,recall,f1-score,support
get_password,0.991667,0.991667,0.991667,120.0
set_up_password,0.992593,0.992593,0.992593,135.0
check_mortgage_payments,0.993289,1.000000,0.996633,148.0
check_loan_payments,1.000000,0.993289,0.996633,149.0
block_card,1.000000,1.000000,1.000000,150.0
make_transfer,1.000000,1.000000,1.000000,147.0
human_agent,1.000000,1.000000,1.000000,149.0
activate_card_international_usage,1.000000,1.000000,1.000000,150.0


,flag,n,accuracy
0,B,3831,0.9992
1,C,3063,0.9997
2,E,391,1.0000
3,I,2694,1.0000
4,L,1920,1.0000
5,M,249,1.0000
6,P,1600,1.0000
7,Q,1924,0.9984
8,W,551,1.0000
9,Z,1960,0.9985


,true,pred,count
0,check_loan_payments,check_mortgage_payments,1
1,get_password,set_up_password,1
2,set_up_password,get_password,1


array([[1.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
      

## Error analysis (fill in 10–20 examples for the report)
Look at wrong predictions on BANKING77: are they mapping problems (our label is debatable) or model problems?

In [5]:
b77_ext = b77_ext.copy(); b77_ext["pred"] = model.predict(b77_ext["text"])
errs = b77_ext[b77_ext["pred"] != b77_ext["gold_intent"]][["b77_intent", "text", "gold_intent", "pred", "confidence"]]
display(errs.sample(min(20, len(errs)), random_state=C.SEED))

,b77_intent,text,gold_intent,pred,confidence
295,card_swallowed,The ATM ate my card.,recover_swallowed_card,find_ATM,high
126,card_payment_fee_charged,Why was I charged for card payment?,check_fees,check_loan_payments,medium
209,passcode_forgotten,I can't recall my passcode and need to reset it.,get_password,check_fees,medium
170,compromised_card,What do I do to stop unauthorized transactions...,block_card,check_current_balance_on_card,medium
130,card_payment_fee_charged,Why did I have to pay extra because I paid wit...,check_fees,human_agent,medium
54,cancel_transfer,"This is URGENT, I typed the wrong payment info...",cancel_transfer,check_loan_payments,high
240,terminate_account,Can you remove my account please?,close_account,check_fees,high
256,terminate_account,I need your help in deleting my account.,close_account,check_fees,high
269,terminate_account,I want my account deleted!,close_account,check_fees,high
94,wrong_amount_of_cash_received,Why did I not get all the cash I asked for?,dispute_ATM_withdrawal,recover_swallowed_card,high
